# Start here: Karakalpak speech recognition

Run this notebook first. Edit the settings below, then call `run_notebooks(train=True)` when ready to train.

The numbered notebooks are readable steps of one workflow. The runner executes them in the same Python session, so variables from one step are available to the next.

[Full beginner guide](../../docs/whisper_training.md).

## 1. Import the tools

These libraries read tables/audio, load Whisper, train it, and measure recognition errors.

In [1]:
from pathlib import Path
import gc
import hashlib
import json
import random
import unicodedata

import numpy as np
import pandas as pd
import soundfile as sf
import torch
from torch.utils.data import DataLoader
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from jiwer import wer, cer

## 2. Find the project folder

Start the terminal or notebook kernel anywhere inside this repository. This loop walks up until it finds the project.

In [2]:
ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").is_file():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Start inside the AI & ML project folder.")
    ROOT = ROOT.parent

NOTEBOOK_DIR = ROOT / "notebooks/whisper"
DATA_DIR = ROOT / "data/karakalpak-combined"
print("Dataset:", DATA_DIR)

Dataset: /Users/macbook/Documents/IT/repos/AI & ML/data/karakalpak-combined


## 3. Settings you can change

Whisper-small is the full pretrained model. Every parameter will be trained. Batch size 1 keeps memory usage low; accumulating 8 batches combines their gradients before an update.

`kk` means Kazakh in Whisper. We keep it as the existing proxy for Karakalpak.

Choose a different `RUN_NAME` for each new training run. Source datasets are only read.

In [3]:
MODEL_ID = "openai/whisper-small"
LANGUAGE = "kk"
SAMPLE_RATE = 16000
SEED = 42

RUN_NAME = "whisper-small-kaa-full-run02"
RUN_DIR = ROOT / "checkpoints" / RUN_NAME

BATCH_SIZE = 1
EPOCHS = 3
LEARNING_RATE = 0.00001
ACCUMULATION_STEPS = 8
VALIDATION_FRACTION = 0.10
MAX_NEW_TOKENS = 444

RUN_TRAINING = False
RUN_FINAL_TEST = False
LOCAL_FILES_ONLY = True  # The model is already cached on this Mac.

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print("Device:", DEVICE)
print("Training output:", RUN_DIR)

Device: mps
Training output: /Users/macbook/Documents/IT/repos/AI & ML/checkpoints/whisper-small-kaa-full-run02


## 4. Run the notebooks with one function

This short function reads each notebook and executes its code cells. `exec` means “run this Python code.”

Checks run before training. An error stops execution immediately. Evaluation runs after training finishes.

In [4]:
def run_notebook(name):
    """Run one notebook in the current Python session."""
    notebook = json.loads((NOTEBOOK_DIR / name).read_text())
    for cell in notebook["cells"]:
        if cell["cell_type"] == "code":
            exec("".join(cell["source"]), globals())

def run_notebooks(train=False):
    global RUN_TRAINING
    RUN_TRAINING = train

    notebooks = [
        "01_data.ipynb",
        "02_model.ipynb",
        "03_batching.ipynb",
        "06_checks.ipynb",
        "04_training.ipynb",
        "05_evaluation.ipynb",
    ]
    for name in notebooks:
        print("\nRunning", name, flush=True)
        run_notebook(name)

## 5. Start the workflow

After running setup, enter `run_notebooks(train=True)` in a new cell to start full training.

Use `run_notebooks(train=False)` to check the data/model without training. Individual stage notebooks use the variables created by previous steps; the runner manages their order.

In [6]:
# Start full training only when ready:
run_notebooks(train=True)

# Check the workflow without training:
# run_notebooks(train=False)


Running 01_data.ipynb
Original train: 3708
Original test: 927
All recordings passed the audio and transcript checks.
Train: 3338
Validation: 370
Test: 927
Example transcript: kese etek ten qońıratqa qaray sozılǵan úlken gúzar jol

Running 02_model.ipynb


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Device: mps
Trainable parameters: 241,734,912
Encoder / decoder layers: 12 / 12

Running 03_batching.ipynb
Audio features: (1, 80, 3000)
Text labels: (1, 33)
Example loss: 4.463428974151611

Running 06_checks.ipynb
Split checks passed.
Settings, batch, and full-model checks passed.

Running 04_training.ipynb
Epoch 1/3 | batch 8/3338 | updates 1 | loss 4.4317
Epoch 1/3 | batch 16/3338 | updates 2 | loss 3.8198
Epoch 1/3 | batch 24/3338 | updates 3 | loss 3.6497
Epoch 1/3 | batch 32/3338 | updates 4 | loss 3.3802
Epoch 1/3 | batch 40/3338 | updates 5 | loss 3.2454
Epoch 1/3 | batch 48/3338 | updates 6 | loss 3.1553
Epoch 1/3 | batch 56/3338 | updates 7 | loss 3.0630
Epoch 1/3 | batch 64/3338 | updates 8 | loss 2.9837
Epoch 1/3 | batch 72/3338 | updates 9 | loss 2.9141
Epoch 1/3 | batch 80/3338 | updates 10 | loss 2.8381
Epoch 1/3 | batch 88/3338 | updates 11 | loss 2.7689
Epoch 1/3 | batch 96/3338 | updates 12 | loss 2.6948
Epoch 1/3 | batch 104/3338 | updates 13 | loss 2.6215
Epoch 1/3 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Validation loss: 0.5626119257905681 | best: 0.5626119257905681
Epoch 2/3 | batch 8/3338 | updates 419 | loss 0.2984
Epoch 2/3 | batch 16/3338 | updates 420 | loss 0.3250
Epoch 2/3 | batch 24/3338 | updates 421 | loss 0.3545
Epoch 2/3 | batch 32/3338 | updates 422 | loss 0.3532
Epoch 2/3 | batch 40/3338 | updates 423 | loss 0.3399
Epoch 2/3 | batch 48/3338 | updates 424 | loss 0.3159
Epoch 2/3 | batch 56/3338 | updates 425 | loss 0.3160
Epoch 2/3 | batch 64/3338 | updates 426 | loss 0.2994
Epoch 2/3 | batch 72/3338 | updates 427 | loss 0.2906
Epoch 2/3 | batch 80/3338 | updates 428 | loss 0.3020
Epoch 2/3 | batch 88/3338 | updates 429 | loss 0.2973
Epoch 2/3 | batch 96/3338 | updates 430 | loss 0.3232
Epoch 2/3 | batch 104/3338 | updates 431 | loss 0.3234
Epoch 2/3 | batch 112/3338 | updates 432 | loss 0.3134
Epoch 2/3 | batch 120/3338 | updates 433 | loss 0.3089
Epoch 2/3 | batch 128/3338 | updates 434 | loss 0.3016
Epoch 2/3 | batch 136/3338 | updates 435 | loss 0.3009
Epoch 2/3 | bat

...

Evaluated 325/370


[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Evaluated 350/370


[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=444) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

WER: 32.447352279697405 %
CER: 7.3108971527839115 %
Reference: kún uyasına sıymay dolanıp qıp qızıl shoqtay janıp batıwdıń az aq aldında tur
Prediction: kún uyasına sıymay dolanıp qıp qızıl shoqtay janıp batıwdıń az aq aldında tur 

Reference: azannan bermaǵan napaqaların taba almay alıstan dińkesi qurıp kiyatırǵanday bes altı ǵarǵa adım jerge bir qonıp kún shıǵarǵa taman ushpaqta
Prediction: azannan bermaǵan napaqaların taba almay alıstan dińkes qurıp kiyatırǵanday bes altı ǵarǵa adım jerde bir qonıp kúnshı ǵarǵa taman ushpaqtı 

Reference: balanıń da arqalaǵan túyinshigi bar
Prediction: balanıń da arqalaǵan túyinshi bar 

Reference: hayal anda sanda artına burılıp qaraydı
Prediction: hayal anda sanda artına burılıp qaraydı 

Reference: bala teńselip júre almay kiyatır
Prediction: bala teńselip júre almay kiyatır 



## Watch a running model

Open [07_progress.ipynb](07_progress.ipynb) in another tab, choose the run name, and run all cells to see its stage, progress bar, loss curve, and completed epochs. This viewer never starts training.